# 05. Final model and the December test

Chosen on November: feature set **E5** (no raw campaign / ad IDs), LightGBM with `num_leaves=127`.
Now we retrain on **May–November** and score **December once**.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

exec(open("04_experiments.py").read().split("# %%\nexperiments")[0])   # same feature lists and params

E5 = RAW_OTHER + GENERAL + PAST_CR + VISITOR + LANDING
CATS = [c for c in ALL_CATS if c in E5]

full = pd.read_parquet(f"{PROCESSED}/features.parquet")
fit_df = full[full["visit_dt"] < "2021-12-01"].copy()
test = full[full["visit_dt"] >= "2021-12-01"].copy()
for c in CATS:
    cats = pd.CategoricalDtype(fit_df[c].unique())
    fit_df[c] = fit_df[c].astype(cats)
    test[c] = test[c].astype(cats)

train 1,102,186 | valid 263,835 | valid CR 2.44%


Number of trees: best iteration on November (128), scaled up ~20% because the training set is bigger.

In [2]:
params = {**PARAMS, "num_leaves": 127, "min_child_samples": 200, "n_estimators": 155}
model = lgb.LGBMClassifier(**params).fit(fit_df[E5], fit_df["target"])
proba = model.predict_proba(test[E5])[:, 1]

auc = roc_auc_score(test["target"], proba)
ap = average_precision_score(test["target"], proba)
print(f"December test — ROC-AUC: {auc:.4f} | PR-AUC: {ap:.4f} (random = {test['target'].mean():.4f})")

December test — ROC-AUC: 0.6767 | PR-AUC: 0.0624 (random = 0.0199)


## What the score means for the business
Rank December sessions by predicted probability and split them into 10 equal groups (deciles).
**Lift** = conversion rate in the group ÷ average conversion rate.

In [3]:
test["score"] = proba
test["decile"] = pd.qcut(test["score"].rank(method="first", ascending=False), 10, labels=range(1, 11))
dec = test.groupby("decile", observed=True)["target"].agg(sessions="size", conversions="sum", cr="mean")
dec["lift"] = (dec["cr"] / test["target"].mean()).round(2)
dec["cum_share_of_conversions"] = (dec["conversions"].cumsum() / dec["conversions"].sum() * 100).round(1)
dec["cr"] = (dec["cr"] * 100).round(2)
dec

,sessions,conversions,cr,lift,cum_share_of_conversions
decile,,,,,
1,36625,1881,5.14,2.58,25.8
2,36624,1106,3.02,1.52,40.9
3,36625,862,2.35,1.18,52.7
4,36624,778,2.12,1.07,63.4
5,36625,729,1.99,1.00,73.4
6,36624,606,1.65,0.83,81.7
7,36624,477,1.30,0.65,88.2
8,36625,405,1.11,0.55,93.8
9,36624,302,0.82,0.41,97.9


## Feature importance (gain)

In [4]:
imp = (pd.Series(model.booster_.feature_importance("gain"), index=E5)
         .sort_values(ascending=False))
(imp / imp.sum() * 100).round(1).head(15)

prev_conversions         15.2
utm_campaign_past_cr     13.6
geo_city                  6.5
utm_source_past_cr        4.9
first_medium              3.4
utm_campaign_past_n       3.3
utm_medium_past_cr        3.2
prev_sessions             2.9
utm_medium                2.6
utm_adcontent_past_cr     2.5
prev_converted            2.4
landing_section           2.3
mean_prev_hits            2.2
days_since_prev           2.1
geo_city_past_n           1.9
dtype: float64

In [5]:
joblib.dump({"model": model, "features": E5, "categories": {c: fit_df[c].cat.categories for c in CATS}},
            "../models/lgbm_conversion.joblib")
test[["session_id", "visit_dt", "target", "score"]].to_parquet(f"{PROCESSED}/test_scores.parquet", index=False)